# 2AFC Constant-Stimuli Psychophysics Analysis

This notebook analyzes a two-alternative forced-choice (2AFC) constant-stimuli experiment.

**Raw data rule:** set `DATA_ROOT` to the main data folder. The notebook treats it as read-only and writes all outputs to `OUTPUT_ROOT`.

**Important fitting rule:** the raw `answer` column is **not** used directly for fitting. Each trial is first converted into `response_comparison_greater` after determining whether the comparison stimulus was object 1 or object 2.

## 0. Configuration

Edit `DATA_ROOT`. Keep `OUTPUT_ROOT` outside the raw data folder.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd() / "analysis"))
import twoafc_psychophysics as pf

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# === EDIT THIS PATH ===
DATA_ROOT = Path(r"C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\debug_experiments")

# Separate output location. Do not place this inside DATA_ROOT.
OUTPUT_ROOT = Path(r"analysis_outputs/2afc_constant_stimuli")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STANDARD_FALLBACK = 85.0          # used only as fallback/tie-break anchor
STANDARD_ABS_TOLERANCE = 0.75     # tolerance for classifying the inferred standard
BOOTSTRAP_N = 500                 # increase for final runs
RANDOM_SEED = 20260508
FIG_DPI = 160

# Optional manual column overrides if automatic detection needs help.
# Example: {"object_1_value": "object_1_stiffness", "answer": "answer"}
MANUAL_COLUMN_OVERRIDES = {}

print("DATA_ROOT:", DATA_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT.resolve())
if str(DATA_ROOT) == "PUT_MAIN_DATA_FOLDER_HERE":
    print("?? Set DATA_ROOT before running the analysis cells.")

DATA_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\debug_experiments
OUTPUT_ROOT: C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Elisheva Shiri Decktor\BGU\Codes\Parallel_Heptics\analysis\analysis_outputs\2afc_constant_stimuli


## 1. File discovery and loading

One folder per subject is expected under `DATA_ROOT`. The code searches each subject folder for answer-like CSV files (`answers.csv`, `answares.csv`, similar), selects the highest-scoring candidate, and saves a discovery audit table.

In [2]:
pf.validate_paths(DATA_ROOT, OUTPUT_ROOT)

file_discovery_summary = pf.discover_answer_files(DATA_ROOT, OUTPUT_ROOT)
pf.save_csv(file_discovery_summary, OUTPUT_ROOT, "file_discovery_summary.csv")
display(file_discovery_summary.head(30))

combined_raw_imported_data = pf.load_selected_subject_csvs(file_discovery_summary)
pf.save_csv(combined_raw_imported_data, OUTPUT_ROOT, "combined_raw_imported_data.csv")
display(combined_raw_imported_data.head())
print("combined raw shape:", combined_raw_imported_data.shape)

,subject_id,subject_folder,candidate_rank,candidate_score,selected,source_file,source_file_name,n_answer_csv_candidates,n_total_csv_files_in_subject_folder,selection_warning
0,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,264,
1,2026_05_04_12_17_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,NaN,NaN,False,,,0,255,no exact answers.csv file found
2,2026_05_04_14_07_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,1.0,37.0,True,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,259,


,subject_id,_subject_folder,_source_file,_source_file_name,_row_in_source,timestamp,pair_number,object_1_finger,object_1_stiffness,object_2_finger,object_2_stiffness,time_to_answer,answer
0,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,0,2026-05-04T10:31:57.847031,1,index,125,index,85,5.484941,0
1,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,2026-05-04T10:32:15.274370,2,index,45,index,85,16.374176,1
2,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,2,2026-05-04T10:32:48.175212,3,index,25,index,85,2.209363,1
3,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,3,2026-05-04T10:32:57.265269,4,index,165,index,85,2.188484,1
4,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,4,2026-05-04T10:33:05.731827,5,index,85,index,145,1.971901,1


combined raw shape: (512, 13)


## 2. Column detection and validation

Detects object values, object fingers, answer, reaction time, trial index, timestamp, and optional block columns. Review the detection table; if needed, set `MANUAL_COLUMN_OVERRIDES` above and rerun.

In [3]:
detected_columns, column_detection_details = pf.detect_columns(combined_raw_imported_data, MANUAL_COLUMN_OVERRIDES)
print(json.dumps(detected_columns, indent=2, ensure_ascii=False))
pf.save_csv(column_detection_details, OUTPUT_ROOT, "column_detection_details.csv")
display(column_detection_details.groupby("target").head(5))

missing_required = [k for k in ["object_1_value", "object_2_value", "answer"] if not detected_columns.get(k)]
if missing_required:
    raise RuntimeError(f"Missing required detected columns: {missing_required}. Add MANUAL_COLUMN_OVERRIDES and rerun.")

{
  "object_1_value": "object_1_stiffness",
  "object_2_value": "object_2_stiffness",
  "object_1_finger": "object_1_finger",
  "object_2_finger": "object_2_finger",
  "answer": "answer",
  "reaction_time": "time_to_answer",
  "trial_index": "pair_number",
  "timestamp": "timestamp",
  "block": null
}


,target,column,normalized,score,manual
0,object_1_value,object_1_stiffness,object_1_stiffness,23,False
1,object_1_value,object_2_stiffness,object_2_stiffness,9,False
2,object_1_value,object_1_finger,object_1_finger,6,False
3,object_1_value,_row_in_source,row_in_source,0,False
4,object_1_value,_source_file,source_file,0,False
8,object_2_value,object_2_stiffness,object_2_stiffness,23,False
9,object_2_value,object_1_stiffness,object_1_stiffness,9,False
10,object_2_value,object_2_finger,object_2_finger,6,False
11,object_2_value,_row_in_source,row_in_source,0,False
12,object_2_value,_source_file,source_file,0,False


## 3. Trial cleaning and canonicalization

Protocol rows and invalid/ambiguous trials are flagged and saved separately. Valid trials receive `response_comparison_greater` for fitting.

Examples:
- standard object 1, comparison object 2 ? response is 1 when `answer == 1`.
- standard object 2, comparison object 1 ? response is 1 when `answer == 0`.

In [4]:
inferred_standard_value, standard_inference_table = pf.infer_standard_value(combined_raw_imported_data, detected_columns, STANDARD_FALLBACK)
print(f"Inferred standard/reference value: {inferred_standard_value:g}")
pf.save_csv(standard_inference_table, OUTPUT_ROOT, "standard_value_inference.csv")
display(standard_inference_table.head(10))

combined_clean_trials, combined_flagged_trials = pf.canonicalize_trials(
    combined_raw_imported_data,
    detected_columns,
    standard_value=inferred_standard_value,
    standard_tolerance=STANDARD_ABS_TOLERANCE,
)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(combined_flagged_trials, OUTPUT_ROOT, "combined_flagged_trials.csv")

print("Clean trials:", combined_clean_trials.shape)
display(combined_clean_trials.head())
print("Flagged trials:", combined_flagged_trials.shape)
display(combined_flagged_trials.head())

Inferred standard/reference value: 85


,candidate_value,count,n_unique_partners,distance_from_fallback,standard_score
0,85,512,8,0.0,1.35000
1,65,64,1,20.0,0.15625
2,105,64,1,20.0,0.15625
3,125,64,1,40.0,0.14375
4,45,64,1,40.0,0.14375
5,25,64,1,60.0,0.13125
6,145,64,1,60.0,0.13125
7,165,64,1,80.0,0.11875
8,5,64,1,80.0,0.11875


Clean trials: (512, 30)


,subject_id,source_file,source_file_name,row_in_source,trial_index_raw,timestamp,block_raw,object_1_value,object_2_value,object_1_finger_raw,object_2_finger_raw,object_1_finger,object_2_finger,standard_value,standard_side,comparison_side,comparison_value,standard_finger,comparison_finger,finger_condition,raw_answer,answer_code,response_comparison_greater,reaction_time,protocol_marker,finger_warning,excluded_from_fit,flag_reason,global_trial_order,answer_chose_object_2
0,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,0,1,2026-05-04T10:31:57.847031,NaN,125,85,index,index,I,I,85.0,object_2,object_1,125.0,I,I,I,0,0,1,5.484941,False,,False,,1,0.0
1,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,1,2,2026-05-04T10:32:15.274370,NaN,45,85,index,index,I,I,85.0,object_2,object_1,45.0,I,I,I,1,1,0,16.374176,False,,False,,2,1.0
2,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,2,3,2026-05-04T10:32:48.175212,NaN,25,85,index,index,I,I,85.0,object_2,object_1,25.0,I,I,I,1,1,0,2.209363,False,,False,,3,1.0
3,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,3,4,2026-05-04T10:32:57.265269,NaN,165,85,index,index,I,I,85.0,object_2,object_1,165.0,I,I,I,1,1,0,2.188484,False,,False,,4,1.0
4,2026_05_04_10_26_single_finger_config_fingers_...,C:\Users\user\BIO MEDICAL ROBOTICS Dropbox\Eli...,answers.csv,4,5,2026-05-04T10:33:05.731827,NaN,85,145,index,index,I,I,85.0,object_1,object_2,145.0,I,I,I,1,1,1,1.971901,False,,False,,5,1.0


Flagged trials: (0, 0)


""


## 4. Block and finger detection

Finger labels are normalized to `I`, `M`, `R`, `P`. Block order is inferred from changes in `finger_condition` within each subject.

In [5]:
combined_clean_trials, block_order_summary = pf.add_block_numbers(combined_clean_trials)
pf.save_csv(combined_clean_trials, OUTPUT_ROOT, "combined_clean_trials.csv")
pf.save_csv(block_order_summary, OUTPUT_ROOT, "block_order_summary.csv")
display(block_order_summary.head(30))
display(pd.crosstab(combined_clean_trials["subject_id"], combined_clean_trials["finger_condition"]))

,subject_id,block_number,finger_condition,first_global_trial_order,n_trials_in_block
0,2026_05_04_10_26_single_finger_config_fingers_...,1,I,1,64
1,2026_05_04_10_26_single_finger_config_fingers_...,2,M,65,64
2,2026_05_04_10_26_single_finger_config_fingers_...,3,P,129,64
3,2026_05_04_10_26_single_finger_config_fingers_...,4,R,193,64
4,2026_05_04_14_07_single_finger_config_fingers_...,1,R,1,64
5,2026_05_04_14_07_single_finger_config_fingers_...,2,P,65,64
6,2026_05_04_14_07_single_finger_config_fingers_...,3,M,129,64
7,2026_05_04_14_07_single_finger_config_fingers_...,4,I,193,64


finger_condition,I,M,P,R
subject_id,,,,
2026_05_04_10_26_single_finger_config_fingers_motor_set_0_013,64,64,64,64
2026_05_04_14_07_single_finger_config_fingers_motor_set_0_019,64,64,64,64


## 5. Quality-control summaries

Flags suspicious cases without silently deleting more data: missing/invalid rows, trial counts per value/finger, non-monotonic curves, apparent error rate, RT outliers, side bias, standard-side bias, and related warnings.

In [6]:
qc_summary = pf.make_qc_summary(combined_clean_trials, combined_flagged_trials)
pf.save_csv(qc_summary, OUTPUT_ROOT, "qc_summary.csv")
display(qc_summary)

,subject_id,finger_condition,n_clean_trials,n_stimulus_levels,min_trials_per_value,median_trials_per_value,side_bias_p_chose_object2,standard_side_p_object2,apparent_error_rate,rt_outlier_count,monotonic_violations,spearman_r,non_monotonic_flag,qc_warnings,n_flagged_rows
0,2026_05_04_10_26_single_finger_config_fingers_...,I,64,8,8,8.0,0.625000,0.5,0.187500,4,1,0.871576,True,non_monotonic_curve,0
1,2026_05_04_10_26_single_finger_config_fingers_...,M,64,8,8,8.0,0.500000,0.5,0.093750,3,0,0.848625,False,,0
2,2026_05_04_10_26_single_finger_config_fingers_...,P,64,8,8,8.0,0.468750,0.5,0.062500,3,0,0.963624,False,,0
3,2026_05_04_10_26_single_finger_config_fingers_...,R,64,8,8,8.0,0.437500,0.5,0.125000,0,0,0.951876,False,,0
4,2026_05_04_14_07_single_finger_config_fingers_...,I,64,8,8,8.0,0.375000,0.5,0.125000,7,1,0.843435,True,non_monotonic_curve,0
5,2026_05_04_14_07_single_finger_config_fingers_...,M,64,8,8,8.0,0.468750,0.5,0.093750,1,0,0.932227,False,,0
6,2026_05_04_14_07_single_finger_config_fingers_...,P,64,8,8,8.0,0.421875,0.5,0.109375,3,1,0.805118,True,non_monotonic_curve,0
7,2026_05_04_14_07_single_finger_config_fingers_...,R,64,8,8,8.0,0.453125,0.5,0.140625,2,1,0.822473,True,non_monotonic_curve,0


## 6. Psychometric aggregation

Aggregates valid trials into binomial counts: `comparison_value`, `n_trials`, `n_comparison_greater`, and observed probability.

In [7]:
psychometric_input_by_subject_finger = pf.make_psychometric_input(combined_clean_trials, ["subject_id", "finger_condition"])
pf.save_csv(psychometric_input_by_subject_finger, OUTPUT_ROOT, "psychometric_input_by_subject_finger.csv")
display(psychometric_input_by_subject_finger.head(30))

,subject_id,finger_condition,comparison_value,n_trials,n_comparison_greater,p_comparison_greater,mean_rt,standard_value
0,2026_05_04_10_26_single_finger_config_fingers_...,I,5.0,8,1,0.125,1.967898,85.0
1,2026_05_04_10_26_single_finger_config_fingers_...,I,25.0,8,1,0.125,1.921485,85.0
2,2026_05_04_10_26_single_finger_config_fingers_...,I,45.0,8,1,0.125,3.665074,85.0
3,2026_05_04_10_26_single_finger_config_fingers_...,I,65.0,8,4,0.500,1.888621,85.0
4,2026_05_04_10_26_single_finger_config_fingers_...,I,105.0,8,6,0.750,2.223199,85.0
5,2026_05_04_10_26_single_finger_config_fingers_...,I,125.0,8,7,0.875,2.256303,85.0
6,2026_05_04_10_26_single_finger_config_fingers_...,I,145.0,8,8,1.000,1.833908,85.0
7,2026_05_04_10_26_single_finger_config_fingers_...,I,165.0,8,6,0.750,1.835421,85.0
8,2026_05_04_10_26_single_finger_config_fingers_...,M,5.0,8,1,0.125,1.956405,85.0
9,2026_05_04_10_26_single_finger_config_fingers_...,M,25.0,8,0,0.000,2.079708,85.0


## Psychometric fitting setup

The notebook attempts Wichmann-lab `psignifit` first when available and compatible. If not, it prints installation guidance and uses the scipy logistic fallback on physical stimulus values.

In [8]:
PSIGNIFIT_AVAILABLE, PSIGNIFIT_STATUS = pf.check_psignifit_available()
print(PSIGNIFIT_STATUS)
if not PSIGNIFIT_AVAILABLE:
    print("Optional psignifit install example: pip install psignifit")
    print("Continuing with scipy logistic fallback fits.")

psignifit import succeeded (version=unknown)


## 7. Subject ? finger psychometric fits

In [9]:
pse_jnd_by_subject_finger = pf.fit_conditions(psychometric_input_by_subject_finger, ["subject_id", "finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(pse_jnd_by_subject_finger, OUTPUT_ROOT, "pse_jnd_by_subject_finger.csv")
display(pse_jnd_by_subject_finger)

,subject_id,finger_condition,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,2026_05_04_10_26_single_finger_config_fingers_...,I,scipy_logistic_fallback,65.000022,2.615906,62.508996,67.740809,0.118447,0.281251,0.125001,0.156250,warning,high_lapse_estimate,64,8,3.714481,64.912851,psignifit_failed: 'dict' object has no attribu...,28.456425,64.868275,1.514155,5.0,165.0
1,2026_05_04_10_26_single_finger_config_fingers_...,M,scipy_logistic_fallback,71.340865,6.345050,64.886991,77.577090,0.045127,0.143978,0.081459,0.062519,ok,,64,8,4.604484,45.715137,psignifit_failed: 'dict' object has no attribu...,18.857569,71.550646,4.739932,5.0,165.0
2,2026_05_04_10_26_single_finger_config_fingers_...,P,scipy_logistic_fallback,85.001967,14.487994,70.513972,99.489961,0.018957,0.000000,0.000000,0.000000,ok,,64,8,2.376749,31.430756,psignifit_failed: 'dict' object has no attribu...,11.715378,85.001967,13.187541,5.0,165.0
3,2026_05_04_10_26_single_finger_config_fingers_...,R,scipy_logistic_fallback,97.908060,16.660839,80.638758,113.960437,0.016727,0.052380,0.052380,0.000000,ok,,64,8,4.657138,44.801500,psignifit_failed: 'dict' object has no attribu...,18.400750,99.470591,14.119713,5.0,165.0
4,2026_05_04_14_07_single_finger_config_fingers_...,I,scipy_logistic_fallback,68.159309,12.231635,57.297444,81.760714,0.023739,0.125885,0.000000,0.125885,ok,,64,8,3.667951,49.335293,psignifit_failed: 'dict' object has no attribu...,20.667647,65.544675,9.014607,5.0,165.0
5,2026_05_04_14_07_single_finger_config_fingers_...,M,scipy_logistic_fallback,105.462704,9.087728,96.754827,114.930284,0.030740,0.058835,0.000000,0.058835,ok,,64,8,1.480541,32.627541,psignifit_failed: 'dict' object has no attribu...,12.313771,104.508206,7.624396,5.0,165.0
6,2026_05_04_14_07_single_finger_config_fingers_...,P,scipy_logistic_fallback,67.893582,2.192818,66.189087,70.574723,0.144520,0.187498,0.000000,0.187498,warning,high_lapse_estimate,64,8,5.274268,44.913288,psignifit_failed: 'dict' object has no attribu...,18.456644,67.268170,1.330669,5.0,165.0
7,2026_05_04_14_07_single_finger_config_fingers_...,R,scipy_logistic_fallback,66.569426,2.111990,64.999818,69.223797,0.155208,0.200000,0.000000,0.200000,warning,high_lapse_estimate,64,8,5.109572,50.686671,psignifit_failed: 'dict' object has no attribu...,21.343336,65.952319,1.208057,5.0,165.0


## 8. Subject-level pooled-across-fingers fits

In [10]:
psychometric_input_subject_pooled = pf.make_psychometric_input(combined_clean_trials, ["subject_id"])
pse_jnd_subject_pooled = pf.fit_conditions(psychometric_input_subject_pooled, ["subject_id"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_subject_pooled, OUTPUT_ROOT, "psychometric_input_subject_pooled.csv")
pf.save_csv(pse_jnd_subject_pooled, OUTPUT_ROOT, "pse_jnd_subject_pooled.csv")
display(pse_jnd_subject_pooled)

,subject_id,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,2026_05_04_10_26_single_finger_config_fingers_...,scipy_logistic_fallback,83.846320,18.806377,64.907015,102.519770,0.014940,0.089252,0.049228,0.040024,ok,,256,8,1.135709,176.761901,psignifit_failed: 'dict' object has no attribu...,84.380950,84.154326,15.238623,5.0,165.0
1,2026_05_04_14_07_single_finger_config_fingers_...,scipy_logistic_fallback,81.476062,14.846047,69.210063,98.902157,0.020453,0.164972,0.000000,0.164972,warning,high_lapse_estimate,256,8,4.494756,168.103971,psignifit_failed: 'dict' object has no attribu...,80.051986,77.548949,9.808152,5.0,165.0


## 9. Group-level fits by finger

Pools all valid trials across subjects within each finger condition.

In [11]:
psychometric_input_group_by_finger = pf.make_psychometric_input(combined_clean_trials, ["finger_condition"])
pse_jnd_group_by_finger = pf.fit_conditions(psychometric_input_group_by_finger, ["finger_condition"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_by_finger, OUTPUT_ROOT, "psychometric_input_group_by_finger.csv")
pf.save_csv(pse_jnd_group_by_finger, OUTPUT_ROOT, "pse_jnd_group_by_finger.csv")
display(pse_jnd_group_by_finger)

,finger_condition,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,I,scipy_logistic_fallback,67.439908,12.412718,56.114462,80.939898,0.023929,0.199488,0.060446,0.139042,ok,,128,8,1.591996,108.964304,psignifit_failed: 'dict' object has no attribu...,50.482152,65.808227,8.282747,5.0,165.0
1,M,scipy_logistic_fallback,91.814315,15.785650,76.232885,107.804184,0.017749,0.078980,0.030823,0.048156,ok,,128,8,4.250173,81.377499,psignifit_failed: 'dict' object has no attribu...,36.688749,91.326140,12.968573,5.0,165.0
2,P,scipy_logistic_fallback,84.569313,14.139258,71.507481,99.785996,0.020103,0.095563,0.000000,0.095563,ok,,128,8,5.355401,75.834989,psignifit_failed: 'dict' object has no attribu...,33.917494,82.210224,11.121880,5.0,165.0
3,R,scipy_logistic_fallback,90.587918,21.065777,71.076549,113.208103,0.013490,0.096881,0.002289,0.094591,ok,,128,8,9.000180,94.531226,psignifit_failed: 'dict' object has no attribu...,43.265613,87.190776,16.561431,5.0,165.0


## 10. Group-level all-pooled fit

In [12]:
all_pooled_trials = combined_clean_trials.copy()
all_pooled_trials["group"] = "all_pooled"
psychometric_input_group_all_pooled = pf.make_psychometric_input(all_pooled_trials, ["group"])
pse_jnd_group_all_pooled = pf.fit_conditions(psychometric_input_group_all_pooled, ["group"], PSIGNIFIT_AVAILABLE)
pf.save_csv(psychometric_input_group_all_pooled, OUTPUT_ROOT, "psychometric_input_group_all_pooled.csv")
pf.save_csv(pse_jnd_group_all_pooled, OUTPUT_ROOT, "pse_jnd_group_all_pooled.csv")
display(pse_jnd_group_all_pooled)

,group,fit_method,pse,jnd,x25,x75,slope_at_pse,lapse_rate,lapse_low,lapse_high,fit_quality,fit_warning,n_trials,n_stimulus_levels,deviance,aic,psignifit_status,neg_log_likelihood,mu,scale,comparison_min,comparison_max
0,all_pooled,scipy_logistic_fallback,84.030688,17.481818,67.714014,102.677651,0.016372,0.123567,0.023104,0.100463,ok,,512,8,4.668056,349.261379,psignifit_failed: 'dict' object has no attribu...,170.630689,81.680418,13.278976,5.0,165.0


## 11. Subject-average group analysis

Computes subject-level probabilities first, then averages subjects so each subject has equal weight. This complements trial-pooled group fits.

In [13]:
subject_average_group_by_finger = pf.subject_average_psychometric(combined_clean_trials, ["finger_condition"])
subject_average_group_all = pf.subject_average_psychometric(combined_clean_trials.assign(group="all_subject_average"), ["group"])
pf.save_csv(subject_average_group_by_finger, OUTPUT_ROOT, "subject_average_group_by_finger.csv")
pf.save_csv(subject_average_group_all, OUTPUT_ROOT, "subject_average_group_all_pooled.csv")
display(subject_average_group_by_finger.head(30))

,finger_condition,comparison_value,n_subjects,mean_p_comparison_greater,sem_p_comparison_greater,total_trials,standard_value
0,I,5.0,2,0.0625,0.0625,16,85.0
1,I,25.0,2,0.0625,0.0625,16,85.0
2,I,45.0,2,0.1250,0.0000,16,85.0
3,I,65.0,2,0.4375,0.0625,16,85.0
4,I,105.0,2,0.8750,0.1250,16,85.0
5,I,125.0,2,0.8125,0.0625,16,85.0
6,I,145.0,2,0.9375,0.0625,16,85.0
7,I,165.0,2,0.8125,0.0625,16,85.0
8,M,5.0,2,0.0625,0.0625,16,85.0
9,M,25.0,2,0.0000,0.0000,16,85.0


## 12. Bootstrap confidence intervals

For each subject ? finger, trials are resampled with replacement and refit. Percentile CIs are saved for PSE and JND.

In [14]:
bootstrap_summary = pf.bootstrap_subject_finger(combined_clean_trials, BOOTSTRAP_N, RANDOM_SEED, PSIGNIFIT_AVAILABLE)
pf.save_csv(bootstrap_summary, OUTPUT_ROOT, "bootstrap_summary.csv")
display(bootstrap_summary)

,subject_id,finger_condition,n_boot_requested,n_boot_success,pse_ci_low,pse_boot_median,pse_ci_high,jnd_ci_low,jnd_boot_median,jnd_ci_high,bootstrap_warning
0,2026_05_04_10_26_single_finger_config_fingers_...,I,500,500,50.827008,65.448356,105.000039,1.071562,10.426692,44.344448,
1,2026_05_04_10_26_single_finger_config_fingers_...,M,500,500,62.731945,69.334893,86.082884,0.915889,1.960594,21.620679,
2,2026_05_04_10_26_single_finger_config_fingers_...,P,500,500,66.663553,88.377848,105.348241,0.878890,4.701864,18.473408,
3,2026_05_04_10_26_single_finger_config_fingers_...,R,500,500,66.957693,102.108192,115.375884,0.918811,13.117120,24.921014,
4,2026_05_04_14_07_single_finger_config_fingers_...,I,500,500,52.931722,65.712301,85.348166,0.927552,9.413275,20.491104,
5,2026_05_04_14_07_single_finger_config_fingers_...,M,500,500,85.554046,105.366180,117.269720,0.878890,5.662429,15.943074,
6,2026_05_04_14_07_single_finger_config_fingers_...,P,500,500,65.303542,68.467991,103.814617,1.013438,1.932681,19.299804,
7,2026_05_04_14_07_single_finger_config_fingers_...,R,500,500,64.644724,67.702645,107.097480,1.048715,2.117648,28.559816,


## 13. Order-effect analysis

Summarizes fatigue/order effects across trial number and inferred block order.

In [15]:
order_effects_summary, order_effects_binned = pf.compute_order_effects(combined_clean_trials)
pf.save_csv(order_effects_summary, OUTPUT_ROOT, "order_effects_summary.csv")
pf.save_csv(order_effects_binned, OUTPUT_ROOT, "order_effects_binned.csv")
display(order_effects_summary)

,subject_id,finger_condition,n_trials,first_half_p_comparison_greater,second_half_p_comparison_greater,first_half_mean_rt,second_half_mean_rt,spearman_response_vs_order,spearman_rt_vs_order,first_block_number
0,2026_05_04_10_26_single_finger_config_fingers_...,I,64,0.53125,0.53125,2.566329,1.831648,-0.076274,-0.472482,1
1,2026_05_04_10_26_single_finger_config_fingers_...,M,64,0.53125,0.53125,2.106116,2.191270,0.035594,-0.184478,2
2,2026_05_04_10_26_single_finger_config_fingers_...,P,64,0.50000,0.50000,1.894839,6.508709,-0.035525,-0.110531,3
3,2026_05_04_10_26_single_finger_config_fingers_...,R,64,0.43750,0.50000,1.848076,1.900917,0.008475,-0.034066,4
4,2026_05_04_14_07_single_finger_config_fingers_...,I,64,0.53125,0.46875,2.639507,2.516276,-0.021992,-0.402152,4
5,2026_05_04_14_07_single_finger_config_fingers_...,M,64,0.43750,0.37500,2.139061,2.111037,-0.075777,-0.191438,3
6,2026_05_04_14_07_single_finger_config_fingers_...,P,64,0.46875,0.37500,2.213495,2.403927,-0.086491,0.088324,2
7,2026_05_04_14_07_single_finger_config_fingers_...,R,64,0.40625,0.43750,2.755174,2.616579,0.055663,-0.341026,1


## 14. Plots and saved outputs

Saves PNG figures: subject ? finger curves, group curves, all-pooled curve, PSE/JND summaries, order effects, side bias, and QC plots. Subject-specific outputs are also written under `OUTPUT_ROOT/subjects/<subject_id>/`.

In [16]:
figure_paths = pf.save_all_figures(
    OUTPUT_ROOT,
    combined_clean_trials,
    qc_summary,
    psychometric_input_by_subject_finger,
    pse_jnd_by_subject_finger,
    psychometric_input_group_by_finger,
    pse_jnd_group_by_finger,
    psychometric_input_group_all_pooled,
    pse_jnd_group_all_pooled,
    order_effects_binned,
    FIG_DPI,
)
print(f"Saved {len(figure_paths)} figure files.")
display(pd.DataFrame({"figure": [str(p) for p in figure_paths]}).head(30))

pf.save_subject_level_outputs(
    OUTPUT_ROOT,
    combined_clean_trials,
    combined_flagged_trials,
    pse_jnd_by_subject_finger,
    pse_jnd_subject_pooled,
)
manifest = pf.analysis_manifest(OUTPUT_ROOT)
display(manifest)
print("Figures folder:", OUTPUT_ROOT / "figures")
print("Subject output folder:", OUTPUT_ROOT / "subjects")

Saved 18 figure files.


,figure
0,analysis_outputs\2afc_constant_stimuli\figures...
1,analysis_outputs\2afc_constant_stimuli\figures...
2,analysis_outputs\2afc_constant_stimuli\figures...
3,analysis_outputs\2afc_constant_stimuli\figures...
4,analysis_outputs\2afc_constant_stimuli\figures...
5,analysis_outputs\2afc_constant_stimuli\figures...
6,analysis_outputs\2afc_constant_stimuli\figures...
7,analysis_outputs\2afc_constant_stimuli\figures...
8,analysis_outputs\2afc_constant_stimuli\figures...
9,analysis_outputs\2afc_constant_stimuli\figures...


,output,exists,path
0,file_discovery_summary.csv,True,analysis_outputs\2afc_constant_stimuli\file_di...
1,combined_raw_imported_data.csv,True,analysis_outputs\2afc_constant_stimuli\combine...
2,combined_clean_trials.csv,True,analysis_outputs\2afc_constant_stimuli\combine...
3,combined_flagged_trials.csv,True,analysis_outputs\2afc_constant_stimuli\combine...
4,qc_summary.csv,True,analysis_outputs\2afc_constant_stimuli\qc_summ...
5,psychometric_input_by_subject_finger.csv,True,analysis_outputs\2afc_constant_stimuli\psychom...
6,pse_jnd_by_subject_finger.csv,True,analysis_outputs\2afc_constant_stimuli\pse_jnd...
7,pse_jnd_subject_pooled.csv,True,analysis_outputs\2afc_constant_stimuli\pse_jnd...
8,pse_jnd_group_by_finger.csv,True,analysis_outputs\2afc_constant_stimuli\pse_jnd...
9,pse_jnd_group_all_pooled.csv,True,analysis_outputs\2afc_constant_stimuli\pse_jnd...


Figures folder: analysis_outputs\2afc_constant_stimuli\figures
Subject output folder: analysis_outputs\2afc_constant_stimuli\subjects


## Interpretation notes

- `PSE` is the physical comparison value where `P(response_comparison_greater) = 0.5`.
- `JND = (x75 - x25) / 2`, in the same units as the stimulus values.
- Review `fit_warning`, `qc_warnings`, and `combined_flagged_trials.csv` before interpreting results.
- Rows excluded from fitting are preserved for audit; no raw data files are modified.